In [ ]:
bert_bool = False

# model_type_str = "multil-i8f32"
# model_name_org_str = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"

# model_type_str = "sonoisa-i8f32"
# model_name_org_str = "sonoisa/sentence-bert-base-ja-mean-tokens-v2"

# model_type_str = "tohoku-bb2-i8f32"
# model_name_org_str = "cl-tohoku/bert-base-japanese-v2"

# model_type_str = "tohoku-bbwwm-i8f32"
# model_name_org_str = "cl-tohoku/bert-base-japanese-whole-word-masking"

model_type_str = "izumil-i8f32"
model_name_org_str = "izumi-lab/bert-small-japanese"

model_name_str = "converted_models/" + model_type_str
filepath_output_str = "scores/score_" + model_type_str + ".txt"

print(f"{bert_bool = }")
print(f"{model_name_str = }")
print(f"{filepath_output_str = }")

In [ ]:
import json

filepath_dataset_eval_str = "../datasets/jsts-v1.1/valid-v1.1.json"

label_str = "label"
sentence1_str = "sentence1"
sentence2_str = "sentence2"


data_eval_dict = dict()
labels_list = list()
sentences1_list = list()
sentences2_list = list()

with open(filepath_dataset_eval_str) as fp:
    for line in fp:
        line_dict = json.loads(line)
        labels_list.append(line_dict[label_str])
        sentences1_list.append(line_dict[sentence1_str])
        sentences2_list.append(line_dict[sentence2_str])

data_eval_dict[label_str] = labels_list
data_eval_dict[sentence1_str] = sentences1_list
data_eval_dict[sentence2_str] = sentences2_list

# print(data_eval_dict)

In [ ]:
import ctranslate2
import numpy as np
import transformers

tokenizer = transformers.AutoTokenizer.from_pretrained(model_name_org_str)
encoder = ctranslate2.Encoder(model_name_str, device="cpu")

In [ ]:
from sentence_transformers import util
from scipy.stats import pearsonr, spearmanr


tokens1_list = tokenizer(data_eval_dict[sentence1_str]).input_ids
tokens2_list = tokenizer(data_eval_dict[sentence2_str]).input_ids

if bert_bool:
    embeddings1_list = np.array(encoder.forward_batch(tokens1_list).pooler_output)
    embeddings2_list = np.array(encoder.forward_batch(tokens2_list).pooler_output)
else:
    embeddings1_list = np.mean(encoder.forward_batch(tokens1_list).last_hidden_state, axis=1)
    embeddings2_list = np.mean(encoder.forward_batch(tokens2_list).last_hidden_state, axis=1)

    
cossim_list = list()
for embedding1, embedding2 in zip(embeddings1_list, embeddings2_list):
    cossim_float = util.cos_sim(embedding1, embedding2).tolist()[0][0]
    cossim_list.append(cossim_float)


pearson_cossim_float, _ = pearsonr(data_eval_dict[label_str], cossim_list)
spearman_cossim_float, _ = spearmanr(data_eval_dict[label_str], cossim_list)

print(f"{pearson_cossim_float = }")
print(f"{spearman_cossim_float = }")


with open(filepath_output_str, "a") as fp:
    fp.write(f"pooling_layer: {bert_bool}\n")
    fp.write(f"pearson_cossim: {pearson_cossim_float}\n")
    fp.write(f"spearman_cossim: {spearman_cossim_float}\n")

In [ ]:
tokens = tokenizer(["これはテストです。", "BERTで埋め込みます。"]).input_ids
# tokens = tokenizer(["これはテストです。"]).input_ids

last_hidden_state = encoder.forward_batch(tokens).last_hidden_state
pooler_output = encoder.forward_batch(tokens).pooler_output

output = np.array(last_hidden_state)
output_mean = np.mean(output, axis=1)

print(f"{tokens = }")
print(f"{type(last_hidden_state) = }")
print(f"{type(pooler_output) = }")
# print(f"{pooler_output.shape = }")
print(f"{pooler_output = }")
print(f"{output.shape = }")
print(f"{output = }")
print(f"{output_mean.shape = }")
print(f"{output_mean = }")
